In [1]:
from de_classes.data_processor import *
from de_classes.spark_factory import SparkFactory
from pyspark.sql import SparkSession, functions as F

In [2]:
measure = "hdfs://localhost:9000/user/student/pollution_parquet"
station = "hdfs://localhost:9000/user/student/Measurement_station_info.csv"
item = "hdfs://localhost:9000/user/student/Measurement_item_info.csv"

In [3]:
spark = SparkFactory("AirQuality-Cleaning").get()

25/08/28 18:24:17 WARN Utils: Your hostname, LAPTOP-5G2QRQLJ.localdomain resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/08/28 18:24:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/28 18:24:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/28 18:24:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
station_raw = DataLoading.read_csv(spark, station)
item_raw = DataLoading.read_csv(spark, item)
measure_raw = spark.read.parquet(measure)

measure_raw = DataCleaning.recast_columns(measure_raw, MEASURE_RENAME_MAP, MEASURE_DTYPES)
station_raw = DataCleaning.recast_columns(station_raw, STATION_RENAME_MAP, STATION_DTYPES)
item_raw = DataCleaning.recast_columns(item_raw, ITEM_RENAME_MAP, ITEM_DTYPES)

station_before = station_raw.count()
item_before = item_raw.count()
measure_before = measure_raw.count()

In [5]:
station = DataCleaning.clean_station(station_raw)
station_after = station.count()
print(station_before, station_after)

25 25


In [6]:
item = DataCleaning.clean_item(item_raw)

item_after = item.count()
print(item_before, item_after)

6 6


In [7]:
measure = DataCleaning.clean_measure(measure_raw)

measure_after = measure.count()
print(measure_before, measure_after)

4032 2758


In [8]:
measure.show()

+-------------------+------------+---------+------------------+-----------------+
|   Measurement_Date|Station_Code|Item_Code|     Average_Value|Instrument_Status|
+-------------------+------------+---------+------------------+-----------------+
|2017-01-01 02:00:00|         103|        6|             0.002|                0|
|2017-01-01 04:00:00|         102|        5|               1.1|                0|
|2017-01-01 04:00:00|         114|        5|               1.3|                0|
|2017-01-01 14:00:00|         102|        5|               0.7|                0|
|2017-01-01 15:00:00|         111|        8|              80.0|                0|
|2017-01-01 02:00:00|         116|        3|             0.063|                0|
|2017-01-01 03:00:00|         101|        5|               1.2|                0|
|2017-01-01 03:00:00|         105|        5|               0.8|                0|
|2017-01-01 04:00:00|         109|        6|             0.002|                0|
|2017-01-01 05:0

In [9]:
errors = DataValidation.detect_measure_errors(measure_raw)
errors.show(20, truncate=False)

error_counts = DataValidation.summarize_error_counts(errors)
error_counts.show()


+-------------------+------------+---------+-------------+-----------------+---------------+
|Measurement_Date   |Station_Code|Item_Code|Average_Value|Instrument_Status|Removal_Reason |
+-------------------+------------+---------+-------------+-----------------+---------------+
|2017-01-01 04:00:00|112         |5        |1.5          |9                |Unreliable data|
|2017-01-01 05:00:00|110         |9        |62.0         |9                |Unreliable data|
|2017-01-01 07:00:00|122         |5        |2.0          |9                |Unreliable data|
|2017-01-01 14:00:00|112         |5        |0.1          |9                |Unreliable data|
|2017-01-01 08:00:00|103         |9        |69.0         |9                |Unreliable data|
|2017-01-01 12:00:00|110         |9        |83.0         |9                |Unreliable data|
|2017-01-01 04:00:00|112         |5        |1.5          |9                |Unreliable data|
|2017-01-01 14:00:00|120         |5        |0.1          |9           

+---------------+-----+
| Removal_Reason|count|
+---------------+-----+
|Unreliable data|   16|
+---------------+-----+



In [10]:
measure.filter(F.col('Average_Value') == 0).show(20, truncate=False)

+----------------+------------+---------+-------------+-----------------+
|Measurement_Date|Station_Code|Item_Code|Average_Value|Instrument_Status|
+----------------+------------+---------+-------------+-----------------+
+----------------+------------+---------+-------------+-----------------+



In [11]:
measure_enriched = DataEnrichment.enrich_measure_with_station_item(measure, station, item)
measure_enriched.limit(20).toPandas()

,Item_Code,Station_Code,Measurement_Date,Average_Value,Instrument_Status,Station_Name,Latitude,Longitude,Address,Item_Name,Unit,Status
0,6,103,2017-01-01 02:00:00,0.002,0,Yongsan-gu,37.540,127.005,"136, Hannam-daero, Yongsan-gu, Seoul, Republic...",O3,ppm,Good
1,5,102,2017-01-01 04:00:00,1.100,0,Jung-gu,37.564,126.975,"15, Deoksugung-gil, Jung-gu, Seoul, Republic o...",CO,ppm,Good
2,5,114,2017-01-01 04:00:00,1.300,0,Nowon-gu,37.659,127.069,"17, Sanggye-ro 23-gil, Nowon-gu, Seoul, Republ...",CO,ppm,Good
3,5,102,2017-01-01 14:00:00,0.700,0,Jung-gu,37.564,126.975,"15, Deoksugung-gil, Jung-gu, Seoul, Republic o...",CO,ppm,Good
4,8,111,2017-01-01 15:00:00,80.000,0,Seongbuk-gu,37.607,127.027,"70, Samyang-ro 2-gil, Seongbuk-gu, Seoul, Repu...",PM10,Mircrogram/m3,Normal
5,3,116,2017-01-01 02:00:00,0.063,0,Gangseo-gu,37.545,126.835,"71, Gangseo-ro 45da-gil, Gangseo-gu, Seoul, Re...",NO2,ppm,Bad
6,5,101,2017-01-01 03:00:00,1.200,0,Jongno-gu,37.572,127.005,"19, Jong-ro 35ga-gil, Jongno-gu, Seoul, Republ...",CO,ppm,Good
7,5,105,2017-01-01 03:00:00,0.800,0,Seodaemun-gu,37.594,126.950,"32, Segeomjeong-ro 4-gil, Seodaemun-gu, Seoul,...",CO,ppm,Good
8,6,109,2017-01-01 04:00:00,0.002,0,Dongdaemun-gu,37.576,127.029,"43, Cheonho-daero 13-gil, Dongdaemun-gu, Seoul...",O3,ppm,Good
9,6,107,2017-01-01 05:00:00,0.002,0,Seongdong-gu,37.542,127.050,"18, Ttukseom-ro 3-gil, Seongdong-gu, Seoul, Re...",O3,ppm,Good


In [12]:
summary, details = DataValidation.validate_enriched_measure(measure_enriched, station, item)

summary.show(truncate=False)
details["Duplicates"].show(20, truncate=False)

+----------------------+-----------+
|Rule_ID               |Validations|
+----------------------+-----------+
|Duplicates            |0          |
|Nulls                 |0          |
|Missing_Station_Info  |0          |
|Missing_Item_Info     |0          |
|Orphan_Station        |0          |
|Orphan_Item           |0          |
|Negative_Values       |0          |
|Bad_Status            |0          |
|Bad_Latitude_Longitude|0          |
+----------------------+-----------+



+----------------+------------+---------+-----+
|Measurement_Date|Station_Code|Item_Code|count|
+----------------+------------+---------+-----+
+----------------+------------+---------+-----+



In [13]:
measure_enriched.repartition(10).write.mode("overwrite").option("header", "true").csv("hdfs://localhost:9000/user/student/processed_data")


In [14]:
spark.stop()